In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [3]:
w1 = pd.read_csv('data/windows/window_1.csv')
w2 = pd.read_csv('data/windows/window_2.csv')
w3 = pd.read_csv('data/windows/window_3.csv')
print(f"Window 1: {w1.shape} | Window 2: {w2.shape} | Window 3: {w3.shape}")

Window 1: (233422, 48) | Window 2: (175067, 48) | Window 3: (175068, 48)


In [4]:
high_null_cols = ['D7', 'D8', 'V257', 'V246', 'V201', 'V200', 'V189', 'V188', 'R_emaildomain']

for df in [w1, w2, w3]:
    for col in high_null_cols:
        df[col + '_missing'] = df[col].isnull().astype(int)

print(f"Missingness indicators created. New shape: {w1.shape}")

Missingness indicators created. New shape: (233422, 57)


In [5]:
num_cols = w1.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c not in ['TransactionID', 'isFraud']]

median_values = w1[num_cols].median()

w1[num_cols] = w1[num_cols].fillna(median_values)
w2[num_cols] = w2[num_cols].fillna(median_values)
w3[num_cols] = w3[num_cols].fillna(median_values)

print(f"Numerical imputation done. Nulls remaining in w1: {w1[num_cols].isnull().sum().sum()}")

Numerical imputation done. Nulls remaining in w1: 0


In [6]:
mode_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain']

for col in mode_cols:
    mode_value = w1[col].mode()[0]
    w1[col] = w1[col].fillna(mode_value)
    w2[col] = w2[col].fillna(mode_value)
    w3[col] = w3[col].fillna(mode_value)

w1['R_emaildomain'] = w1['R_emaildomain'].fillna('Unknown')
w2['R_emaildomain'] = w2['R_emaildomain'].fillna('Unknown')
w3['R_emaildomain'] = w3['R_emaildomain'].fillna('Unknown')

print("Categorical imputation done.")

Categorical imputation done.


In [7]:
m_cols = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
mapping = {'T': 1, 'F': 0}

for col in m_cols:
    w1[col] = w1[col].map(mapping).fillna(0).astype(int)
    w2[col] = w2[col].map(mapping).fillna(0).astype(int)
    w3[col] = w3[col].map(mapping).fillna(0).astype(int)

print("M columns encoded.")

M columns encoded.


In [8]:
cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']
global_mean = w1['isFraud'].mean()

for col in cat_cols:
    encoding_map = w1.groupby(col)['isFraud'].mean()
    w1[col] = w1[col].map(encoding_map).fillna(global_mean)
    w2[col] = w2[col].map(encoding_map).fillna(global_mean)
    w3[col] = w3[col].map(encoding_map).fillna(global_mean)

print("Target encoding done.")

Target encoding done.


In [9]:
feature_cols = [c for c in w1.columns if c not in ['TransactionID', 'isFraud']]

zero_var_cols = [col for col in feature_cols if w1[col].nunique() == 1]
print(f"Zero variance columns: {zero_var_cols}")

w1 = w1.drop(columns=zero_var_cols)
w2 = w2.drop(columns=zero_var_cols)
w3 = w3.drop(columns=zero_var_cols)

feature_cols = [c for c in w1.columns if c not in ['TransactionID', 'isFraud']]
print(f"Features after drop: {len(feature_cols)}")

Zero variance columns: ['M4']
Features after drop: 54


In [10]:
X_train = w1[feature_cols]
X_w2 = w2[feature_cols]
X_w3 = w3[feature_cols]

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_w2_scaled = pd.DataFrame(scaler.transform(X_w2), columns=feature_cols)
X_w3_scaled = pd.DataFrame(scaler.transform(X_w3), columns=feature_cols)

print(f"Train: {X_train_scaled.shape} | W2: {X_w2_scaled.shape} | W3: {X_w3_scaled.shape}")

Train: (233422, 54) | W2: (175067, 54) | W3: (175068, 54)


In [11]:
y_train = w1['isFraud']
sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train_scaled, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {pd.Series(y_train_smote).value_counts().to_dict()}")
print(f"Final train shape: {X_train_smote.shape}")


Before SMOTE: {0: 226024, 1: 7398}
After SMOTE: {0: 226024, 1: 226024}
Final train shape: (452048, 54)


In [12]:
os.makedirs('data/processed', exist_ok=True)

X_train_final = pd.DataFrame(X_train_smote, columns=feature_cols)
X_train_final['isFraud'] = y_train_smote

X_w2_scaled['isFraud'] = w2['isFraud'].values
X_w3_scaled['isFraud'] = w3['isFraud'].values

X_train_final.to_csv('data/processed/train_processed.csv', index=False)
X_w2_scaled.to_csv('data/processed/w2_processed.csv', index=False)
X_w3_scaled.to_csv('data/processed/w3_processed.csv', index=False)
print("All processed files saved.")

All processed files saved.


In [13]:
import os
base_path = os.path.abspath('../data/processed')
print(base_path)

c:\fraud-drift-detection\data\processed


In [14]:
import joblib
import os

os.makedirs("../artifacts", exist_ok=True)

joblib.dump(scaler, "../artifacts/scaler.pkl")
joblib.dump(feature_cols, "../artifacts/feature_columns.pkl")

print("Scaler and feature columns saved successfully.")

Scaler and feature columns saved successfully.
